# Notebook 02 — Graph Theory for Biological Networks

**Module:** 12 — Systems and Network Biology  
**Tier:** 2 — Working competence  
**Notebook:** 02 of 12  
**Time estimate:** 75 minutes

> A protein–protein interaction network is just a graph. Once you see that,
> 150 years of graph theory become immediately applicable to cell biology.

---
## Step 1 — Motivation

Biological networks — PPIs, gene regulatory networks, metabolic networks, ecological
food webs — are all graphs. Graph theory provides the vocabulary and the algorithms
to characterize them. This notebook builds that vocabulary from first principles.

---
## Step 2 — Intuition

A **graph** $G = (V, E)$ is a set of **vertices** (nodes) $V$ and **edges** $E \subseteq V \times V$.

- **Undirected:** $(u,v) = (v,u)$ — PPI networks (symmetric binding)
- **Directed:** $(u,v) \neq (v,u)$ — gene regulatory networks (A activates B ≠ B activates A)
- **Weighted:** each edge has a weight — coexpression networks (Pearson r), STRING (confidence score)

**Degree** of node $v$: $k_v$ = number of edges incident to $v$.  
In directed graphs: in-degree ($k^{in}$) and out-degree ($k^{out}$).

**Path:** sequence of vertices connected by edges.  
**Shortest path:** path minimizing number of edges (unweighted) or total weight (weighted).  
**Connected component:** maximal subset of nodes with paths between all pairs.

---
## Step 3 — Biological Background

**Protein–Protein Interaction (PPI) networks:**
- Nodes: proteins; Edges: physical interaction (binding)
- Typically undirected (binding is symmetric)
- Sources: STRING (confidence-scored), BioGRID (curated), IntAct
- Human interactome: estimated ~300,000–500,000 interactions

**Gene Regulatory Networks (GRNs):**
- Nodes: genes/TFs; Edges: regulation (activation/repression)
- Directed and signed (+ = activates, − = represses)
- Built from ChIP-seq, reporter assays, perturbation data

**Metabolic Networks:**
- Two representations: (1) bipartite (metabolites + reactions) or (2) metabolite-metabolite (sharing a reaction)
- Directed (substrates → products)
- Genome-scale metabolic models (GEMs): E. coli iJO1366 has 1805 genes, 2583 reactions, 1805 metabolites

**Graph representation choice matters:**
- **Adjacency matrix** $A$: $A_{ij} = 1$ if $(i,j) \in E$, else 0. Dense storage: $O(n^2)$. Fast for matrix operations.
- **Edge list:** list of $(u,v)$ pairs. Sparse storage: $O(m)$. Good for large sparse networks.
- **Adjacency list:** dict of sets. Sparse storage, fast neighbor lookup: $O(\deg(v))$.

---
## Step 4 — Mathematical Explanation

**Adjacency matrix** for undirected graph with $n$ nodes:
$$A_{ij} = A_{ji} = \begin{cases} 1 & \text{if } (i,j) \in E \\ 0 & \text{otherwise} \end{cases}$$

**Degree:** $k_i = \sum_j A_{ij}$ (row sum)

**Degree sequence:** sorted list of degrees. The degree distribution $P(k)$ is the fraction of nodes with degree $k$.

**Handshaking lemma:** $\sum_i k_i = 2|E|$ — every edge contributes 2 to the total degree sum.

**Adjacency matrix powers:** $(A^m)_{ij}$ = number of walks of length $m$ from $i$ to $j$.  
This is why eigenvalues of $A$ (the **spectrum**) encode structural properties.

**Laplacian matrix:** $L = D - A$, where $D$ is the diagonal degree matrix ($D_{ii} = k_i$).  
The number of zero eigenvalues of $L$ = number of connected components.

In [ ]:
# Step 6 — Python: Graph representations from scratch
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, deque

# ---- Build a small graph from scratch ----
# Represent a PPI-like network with 10 nodes, 14 edges
# Edge list representation
edges = [
    (0,1),(0,2),(0,3),(1,2),(1,4),(2,5),(3,4),(3,6),
    (4,5),(4,7),(5,8),(6,7),(7,8),(7,9)
]
n_nodes = 10
n_edges = len(edges)

# ---- 1. Adjacency matrix ----
def build_adjacency_matrix(edges, n):
    A = np.zeros((n, n), dtype=int)
    for u, v in edges:
        A[u, v] = 1
        A[v, u] = 1  # undirected
    return A

A = build_adjacency_matrix(edges, n_nodes)
print('Adjacency matrix (10×10):')
print(A)

# ---- 2. Degree sequence ----
degrees = A.sum(axis=1)
print(f'\nDegree sequence: {sorted(degrees, reverse=True)}')
print(f'Sum of degrees: {degrees.sum()} = 2 × {n_edges} edges (handshaking lemma)')

# ---- 3. Adjacency list ----
def build_adjacency_list(edges, n):
    adj = defaultdict(set)
    for u, v in edges:
        adj[u].add(v)
        adj[v].add(u)
    return dict(adj)

adj = build_adjacency_list(edges, n_nodes)
print(f'\nNeighbors of node 4: {sorted(adj[4])}')
print(f'Degree of node 4: {len(adj[4])}')

# ---- 4. BFS from scratch: shortest paths from source node ----
def bfs_shortest_paths(adj, source, n):
    dist = {v: -1 for v in range(n)}
    dist[source] = 0
    queue = deque([source])
    while queue:
        u = queue.popleft()
        for v in adj.get(u, []):
            if dist[v] == -1:
                dist[v] = dist[u] + 1
                queue.append(v)
    return dist

dist_from_0 = bfs_shortest_paths(adj, 0, n_nodes)
print(f'\nBFS shortest paths from node 0: {dist_from_0}')

# ---- 5. Laplacian and connected components ----
D = np.diag(degrees)
L = D - A
eigenvalues = np.sort(np.linalg.eigvalsh(L))
n_components = np.sum(np.abs(eigenvalues) < 1e-10)
print(f'\nLaplacian eigenvalues (smallest 5): {eigenvalues[:5].round(4)}')
print(f'Number of connected components: {n_components}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: Network visualization using spring layout (manual implementation)
# Use adjacency matrix for a crude force-directed layout
def spring_layout(A, iterations=200, seed=42):
    rng = np.random.default_rng(seed)
    n = len(A)
    pos = rng.random((n, 2)) * 2 - 1  # init in [-1,1]^2
    k = 1.0 / np.sqrt(n)  # optimal distance
    for _ in range(iterations):
        disp = np.zeros((n, 2))
        for u in range(n):
            for v in range(n):
                if u == v:
                    continue
                delta = pos[u] - pos[v]
                dist = max(np.linalg.norm(delta), 1e-3)
                disp[u] += delta / dist * (k**2 / dist)  # repulsion
        for u, v in zip(*np.where(np.triu(A))):
            delta = pos[u] - pos[v]
            dist = max(np.linalg.norm(delta), 1e-3)
            force = delta / dist * dist**2 / k
            disp[u] -= force
            disp[v] += force
        pos += disp * 0.1
        # Normalize
        pos -= pos.mean(0)
        max_d = np.abs(pos).max()
        if max_d > 0:
            pos /= max_d
    return pos

pos = spring_layout(A)
ax = axes[0]
for u, v in edges:
    ax.plot([pos[u,0], pos[v,0]], [pos[u,1], pos[v,1]], 'grey', lw=0.8, alpha=0.5, zorder=1)
sc = ax.scatter(pos[:,0], pos[:,1], c=degrees, cmap='Reds', s=200, zorder=2, edgecolors='black', lw=0.5)
for i in range(n_nodes):
    ax.text(pos[i,0]+0.02, pos[i,1]+0.02, str(i), fontsize=7)
plt.colorbar(sc, ax=ax, label='Degree')
ax.set_title('A. Graph visualization\n(color = degree)')
ax.axis('off')

# Panel B: Adjacency matrix heatmap
ax = axes[1]
im = ax.imshow(A, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_title('B. Adjacency matrix A')
ax.set_xlabel('Node'); ax.set_ylabel('Node')
ax.set_xticks(range(n_nodes))
ax.set_yticks(range(n_nodes))

# Panel C: Degree distribution (bar)
ax = axes[2]
deg_vals, deg_counts = np.unique(degrees, return_counts=True)
deg_prob = deg_counts / n_nodes
ax.bar(deg_vals, deg_prob, color='steelblue', edgecolor='black', width=0.4)
ax.set_xlabel('Degree k')
ax.set_ylabel('P(k)')
ax.set_title(f'C. Degree distribution\n(N={n_nodes}, M={n_edges})')
ax.set_xticks(deg_vals)

plt.suptitle('Module 12 NB02: Graph Theory for Biological Networks', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('graph_theory.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Add a node 10 connected to nodes 0 and 9 only. Update the adjacency matrix and
   verify the handshaking lemma still holds.
2. Implement DFS (depth-first search) from scratch. Which nodes are reachable from node 9?
3. Compute the adjacency matrix power $A^2$. What does $A^2_{ij}$ count?
4. For a directed graph where edge $(u,v)$ means "gene $u$ activates gene $v$," what
   changes in the adjacency matrix compared to the undirected case?

---
## Step 10 — Quiz

1. What is the adjacency matrix of an undirected graph? Is it symmetric?
2. What does the handshaking lemma say?
3. What information does $(A^2)_{ij}$ contain?
4. How do you detect connected components from the Laplacian eigenvalues?
5. Which graph representation is most memory-efficient for a sparse network?

---
## Step 12 — Reflection

> *[In one paragraph: explain what information is captured in the adjacency matrix
> that is NOT captured in the degree sequence alone, using a biological network example.]*

---
*Next: `03_network_topology_and_properties.ipynb`*